# 02. 퍼널 분석 — Receive → View → Complete 구조 정의

**분석 목적:**  
오퍼가 실제 구매에 영향을 주는지 확인하기 위한 구조를 정의한다.  
이벤트 로그에서 오퍼 1건당 고객 반응(수신 → 확인 → 구매)을 추적 가능한 단위로 통합한다.

**데이터:** `prep_master_table.csv`  
**핵심 문제:**

이벤트 로그는 received / viewed / completed가 행으로 분산  
→ 오퍼 1건에 대한 고객 반응을 추적 불가  
→ inst_id 도입으로 퍼널을 1행으로 통합

**퍼널 지표 4개:**

| 지표 | 정의 | 의미 |
|------|------|------|
| view_rate | received 대비 viewed 비율 | 오퍼 인지율 |
| conversion_rate | received 대비 completed 비율 | 전체 전환율 |
| view_to_complete_rate | viewed 대비 completed 비율 | 설득력 |
| no_view_complete_rate | viewed 없이 completed 비율 | **우연완료율** |

---

## 0. 환경 설정

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# 전처리 완료된 마스터 테이블 로드
# 01_data_preparation.ipynb 실행 후 생성된 파일 사용
mt = pd.read_csv("../data/processed/prep_master_table.csv")

mt.head()

,person,event,time,amount,offer_id,reward,gender,age,became_member_on,income,...,reward_offer,channels,difficulty,duration,offer_type,email,mobile,social,web,duration_hr
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,0,NaN,9b98b8c7a33c4b65b9aebfe6a799e6d9,NaN,F,75,2017-05-09,100000.0,...,5.0,"['web', 'email', 'mobile']",5.0,7.0,bogo,1.0,1.0,0.0,1.0,168.0
1,e2127556f4f64592b11af22de27a7932,offer received,0,NaN,2906b810c7d4411798c6938adc9daaa5,NaN,M,68,2018-04-26,70000.0,...,2.0,"['web', 'email', 'mobile']",10.0,7.0,discount,1.0,1.0,0.0,1.0,168.0
2,389bc3fa690240e798340f5a15918d5c,offer received,0,NaN,f19421c1d4aa40978ebb69ca19b0e20d,NaN,M,65,2018-02-09,53000.0,...,5.0,"['web', 'email', 'mobile', 'social']",5.0,5.0,bogo,1.0,1.0,1.0,1.0,120.0
3,2eeac8d8feae4a8cad5a6af0499a211d,offer received,0,NaN,3f207df678b143eea3cee63160fa8bed,NaN,M,58,2017-11-11,51000.0,...,0.0,"['web', 'email', 'mobile']",0.0,4.0,informational,1.0,1.0,0.0,1.0,96.0
4,aa4862eba776480b8bb9c68455b8c2e1,offer received,0,NaN,0b1e1539f2cc45b7b9fa7c272da2e1d7,NaN,F,61,2017-09-11,57000.0,...,5.0,"['web', 'email']",20.0,10.0,discount,1.0,0.0,0.0,1.0,240.0


## 1. 왜 퍼널 재정의가 필요한가?

기존 이벤트 로그의 구조적 문제:

```
person  | event            | time | offer_id
------  | -----            | ---- | --------
A       | offer received   | 0    | X001
A       | offer viewed     | 12   | X001
A       | transaction      | 24   | -
A       | offer completed  | 24   | X001
```

→ 한 오퍼에 대한 "수신 → 확인 → 완료"가 3개 행으로 분산  
→ 같은 고객이 동일 오퍼를 여러 번 받을 수 있어 중복 집계 위험

**해결:** 오퍼 received 1회 = inst_id 1개 = 분석 단위 1행

---

## 2. offer inst_id 생성 (received 기준)

동일 고객이 같은 오퍼를 여러 번 받을 경우 발송 순서별로 번호 부여

In [3]:
# offer lifecycle 생성 STEP 1
# received 기준 오퍼 인스턴스(inst_id) 생성

# 퍼널 분석에 필요한 오퍼 이벤트만 사용
offer_events = mt[
    mt['event'].isin([
        'offer received',
        'offer viewed',
        'offer completed'
    ])
].copy()

# 사람 + 오퍼 + 시간 기준 정렬
offer_events = offer_events.sort_values(
    ['person', 'offer_id', 'time']
)

# offer received가 나올 때마다 inst_id를 1씩 증가
# → received 1회 = 오퍼 인스턴스 1개
offer_events['inst_id'] = (
    offer_events['event']
    .eq('offer received')
    .groupby([offer_events['person'], offer_events['offer_id']])
    .cumsum()
)

# received 없이 발생한 viewed / completed 이벤트 제거
offer_events = offer_events[
    offer_events['inst_id'] > 0
].copy()

offer_events.head()

,person,event,time,amount,offer_id,reward,gender,age,became_member_on,income,...,channels,difficulty,duration,offer_type,email,mobile,social,web,duration_hr,inst_id
219501,0009655768c64bdeb2e877511632db8f,offer received,576,NaN,2906b810c7d4411798c6938adc9daaa5,NaN,M,33,2017-04-21,72000.0,...,"['web', 'email', 'mobile']",10.0,7.0,discount,1.0,1.0,0.0,1.0,168.0,1
229124,0009655768c64bdeb2e877511632db8f,offer completed,576,NaN,2906b810c7d4411798c6938adc9daaa5,2.0,M,33,2017-04-21,72000.0,...,"['web', 'email', 'mobile']",10.0,7.0,discount,1.0,1.0,0.0,1.0,168.0,1
100679,0009655768c64bdeb2e877511632db8f,offer received,336,NaN,3f207df678b143eea3cee63160fa8bed,NaN,M,33,2017-04-21,72000.0,...,"['web', 'email', 'mobile']",0.0,4.0,informational,1.0,1.0,0.0,1.0,96.0,1
123887,0009655768c64bdeb2e877511632db8f,offer viewed,372,NaN,3f207df678b143eea3cee63160fa8bed,NaN,M,33,2017-04-21,72000.0,...,"['web', 'email', 'mobile']",0.0,4.0,informational,1.0,1.0,0.0,1.0,96.0,1
49596,0009655768c64bdeb2e877511632db8f,offer received,168,NaN,5a8bc65990b245e5a138643cd4eb9837,NaN,M,33,2017-04-21,72000.0,...,"['email', 'mobile', 'social']",0.0,3.0,informational,1.0,1.0,1.0,0.0,72.0,1


## 3. offer window 생성 (received ~ 만료)

오퍼에는 유효기간(duration)이 존재  
→ received_time + duration_hr = expiry_time  
→ 유효기간 내 반응만 마케팅 기여로 인정

In [4]:
# offer lifecycle 생성 STEP 2
# received 기준 오퍼 윈도우 생성

# received 이벤트만 분리
received = offer_events[
    offer_events['event'] == 'offer received'
].copy()

received = received.rename(
    columns={'time': 'received_time'}
)

# received 1회(inst_id) 기준 오퍼 윈도우 테이블
offer_windows = (
    received
    .groupby(['person', 'offer_id', 'inst_id'], as_index=False)
    .agg(
        received_time=('received_time', 'min'),
        duration_hr=('duration_hr', 'first'),
        offer_type=('offer_type', 'first'),
        email=('email', 'first'),
        mobile=('mobile', 'first'),
        social=('social', 'first'),
        web=('web', 'first')
    )
)

# 오퍼 종료 시간 계산
offer_windows['end_time'] = (
    offer_windows['received_time'] +
    offer_windows['duration_hr']
)

offer_windows.head()

,person,offer_id,inst_id,received_time,duration_hr,offer_type,email,mobile,social,web,end_time
0,0009655768c64bdeb2e877511632db8f,2906b810c7d4411798c6938adc9daaa5,1,576,168.0,discount,1.0,1.0,0.0,1.0,744.0
1,0009655768c64bdeb2e877511632db8f,3f207df678b143eea3cee63160fa8bed,1,336,96.0,informational,1.0,1.0,0.0,1.0,432.0
2,0009655768c64bdeb2e877511632db8f,5a8bc65990b245e5a138643cd4eb9837,1,168,72.0,informational,1.0,1.0,1.0,0.0,240.0
3,0009655768c64bdeb2e877511632db8f,f19421c1d4aa40978ebb69ca19b0e20d,1,408,120.0,bogo,1.0,1.0,1.0,1.0,528.0
4,0009655768c64bdeb2e877511632db8f,fafdcd668e3743c1bb461111dcafc2a4,1,504,240.0,discount,1.0,1.0,1.0,1.0,744.0


## 4. viewed / completed 시간 연결

inst_id 기준으로 최초 viewed/completed 시점을 연결  
→ 같은 오퍼를 여러 번 본 경우 첫 번째만 사용

In [5]:
# offer lifecycle 생성 STEP 3
# viewed / completed 시간 연결

# viewed 이벤트
viewed = offer_events[
    offer_events['event'] == 'offer viewed'
][['person', 'offer_id', 'inst_id', 'time']] \
.rename(columns={'time': 'viewed_time'})

# completed 이벤트
completed = offer_events[
    offer_events['event'] == 'offer completed'
][['person', 'offer_id', 'inst_id', 'time']] \
.rename(columns={'time': 'completed_time'})

# inst_id 기준으로 가장 빠른 viewed / completed 연결
offer_windows = (
    offer_windows
    .merge(
        viewed.groupby(
            ['person', 'offer_id', 'inst_id'], as_index=False
        )['viewed_time'].min(),
        on=['person', 'offer_id', 'inst_id'],
        how='left'
    )
    .merge(
        completed.groupby(
            ['person', 'offer_id', 'inst_id'], as_index=False
        )['completed_time'].min(),
        on=['person', 'offer_id', 'inst_id'],
        how='left'
    )
)

offer_windows.head()

,person,offer_id,inst_id,received_time,duration_hr,offer_type,email,mobile,social,web,end_time,viewed_time,completed_time
0,0009655768c64bdeb2e877511632db8f,2906b810c7d4411798c6938adc9daaa5,1,576,168.0,discount,1.0,1.0,0.0,1.0,744.0,NaN,576.0
1,0009655768c64bdeb2e877511632db8f,3f207df678b143eea3cee63160fa8bed,1,336,96.0,informational,1.0,1.0,0.0,1.0,432.0,372.0,NaN
2,0009655768c64bdeb2e877511632db8f,5a8bc65990b245e5a138643cd4eb9837,1,168,72.0,informational,1.0,1.0,1.0,0.0,240.0,192.0,NaN
3,0009655768c64bdeb2e877511632db8f,f19421c1d4aa40978ebb69ca19b0e20d,1,408,120.0,bogo,1.0,1.0,1.0,1.0,528.0,456.0,414.0
4,0009655768c64bdeb2e877511632db8f,fafdcd668e3743c1bb461111dcafc2a4,1,504,240.0,discount,1.0,1.0,1.0,1.0,744.0,540.0,528.0


## 5. 유효 반응 검증

**제거 조건:** viewed 또는 completed가 received 이전이거나 만료 이후  
→ 데이터 오염 방지 · 실제 마케팅 반응만 추출

In [6]:
# offer lifecycle 생성 STEP 4
# 시간 검증 및 퍼널 상태 컬럼 생성

# viewed는 received 이후 & 유효기간 내만 인정
offer_windows['viewed_valid'] = (
    offer_windows['viewed_time']
    .between(
        offer_windows['received_time'],
        offer_windows['end_time'],
        inclusive='both'
    )
)

offer_windows.loc[
    ~offer_windows['viewed_valid'],
    'viewed_time'
] = np.nan

# completed도 동일하게 검증
offer_windows['completed_valid'] = (
    offer_windows['completed_time']
    .between(
        offer_windows['received_time'],
        offer_windows['end_time'],
        inclusive='both'
    )
)

offer_windows.loc[
    ~offer_windows['completed_valid'],
    'completed_time'
] = np.nan

# 퍼널 상태 컬럼
offer_windows['is_viewed'] = offer_windows['viewed_time'].notna()
offer_windows['is_completed'] = offer_windows['completed_time'].notna()

# 확인 없이 완료 → 마케팅 비기여(우연/체리 가능성)
offer_windows['completed_without_view'] = (offer_windows['is_completed'] & ~offer_windows['is_viewed'])
# 확인 후 완료 → 정상적인 마케팅 전환
offer_windows['viewed_then_completed'] = (
    offer_windows['is_viewed'] &
    offer_windows['is_completed'] &
    (offer_windows['viewed_time'] < offer_windows['completed_time'])
)

offer_windows.head()

,person,offer_id,inst_id,received_time,duration_hr,offer_type,email,mobile,social,web,end_time,viewed_time,completed_time,viewed_valid,completed_valid,is_viewed,is_completed,completed_without_view,viewed_then_completed
0,0009655768c64bdeb2e877511632db8f,2906b810c7d4411798c6938adc9daaa5,1,576,168.0,discount,1.0,1.0,0.0,1.0,744.0,NaN,576.0,False,True,False,True,True,False
1,0009655768c64bdeb2e877511632db8f,3f207df678b143eea3cee63160fa8bed,1,336,96.0,informational,1.0,1.0,0.0,1.0,432.0,372.0,NaN,True,False,True,False,False,False
2,0009655768c64bdeb2e877511632db8f,5a8bc65990b245e5a138643cd4eb9837,1,168,72.0,informational,1.0,1.0,1.0,0.0,240.0,192.0,NaN,True,False,True,False,False,False
3,0009655768c64bdeb2e877511632db8f,f19421c1d4aa40978ebb69ca19b0e20d,1,408,120.0,bogo,1.0,1.0,1.0,1.0,528.0,456.0,414.0,True,True,True,True,False,False
4,0009655768c64bdeb2e877511632db8f,fafdcd668e3743c1bb461111dcafc2a4,1,504,240.0,discount,1.0,1.0,1.0,1.0,744.0,540.0,528.0,True,True,True,True,False,False


## 6. 퍼널 지표 생성

inst_id 기준 1행으로 집계된 최종 퍼널 테이블

In [7]:
# STEP 5. 퍼널 분석용 지표 4개 생성

offer_metrics = offer_windows.copy()

offer_metrics['received'] = 1


# 퍼널 상태를 숫자로 변환
offer_metrics['viewed'] = offer_metrics['is_viewed'].astype(int)
offer_metrics['completed'] = offer_metrics['is_completed'].astype(int)
offer_metrics['completed_wo_view'] = (
    offer_metrics['completed_without_view'].astype(int)
)


#퍼널 지표 4개 생성

# 1)오퍼 확인율 - 오퍼를 받은 사람 중 확인한 비율
offer_metrics['view_rate'] = (
    offer_metrics['viewed'] / offer_metrics['received']
)

# 2)수신 대비 완료율 (= 전환율)오퍼를 받은 사람 중 실제 완료한 비율
offer_metrics['conversion_rate'] = (
    offer_metrics['completed'] / offer_metrics['received']
)

# 3)확인 대비 완료율 (보조) - 오퍼를 확인한 사람 중 완료한 비율
offer_metrics['viewed_to_completed_rate'] = (
    offer_metrics['viewed_then_completed'].astype(int) /
    offer_metrics['viewed'].replace(0, np.nan)
)

# 4)확인 없이 완료 비율 (우연/체리 가능성,마케팅 비기여) - 오퍼를 보지않고 완료한 비율
offer_metrics['completed_wo_view_rate'] = (
    offer_metrics['completed_wo_view'] / offer_metrics['received']
)
# 고객 속성 붙이기 (소득, 나이, 가입기간)
customer_profile = (
     mt[['person', 'income_g', 'age_g', 'membership_g']]
    .drop_duplicates()
)

offer_metrics = offer_metrics.merge(
    customer_profile,
    on='person',
    how='left'
)
# channels (list) → 문자열로 변환
mt['channels'] = mt['channels'].apply(
    lambda x: '+'.join(sorted(x)) if isinstance(x, list) else x
)

#채널 정보 merge 
offer_channel = (mt[['person', 'offer_id', 'channels']].drop_duplicates())
offer_metrics = offer_metrics.merge(offer_channel,on=['person', 'offer_id'],how='left')
#reward 정보 merge
offer_reward = (mt[['offer_id', 'reward']].drop_duplicates())
offer_metrics = offer_metrics.merge(offer_reward,on='offer_id',how='left')

# 결과 확인
offer_metrics[
    [
        'received',
        'view_rate',
        'conversion_rate',
        'viewed_to_completed_rate',
        'completed_wo_view_rate'
    ]
].head()

,received,view_rate,conversion_rate,viewed_to_completed_rate,completed_wo_view_rate
0,1,0.0,1.0,NaN,1.0
1,1,0.0,1.0,NaN,1.0
2,1,1.0,0.0,0.0,0.0
3,1,1.0,0.0,0.0,0.0
4,1,1.0,1.0,0.0,0.0


In [8]:
# offer_metrics 저장 — 03~05 노트북에서 공통 로드
offer_metrics.to_csv('../data/processed/offer_metrics.csv', index=False)
print(f'offer_metrics 저장 완료: {offer_metrics.shape}')

offer_metrics 저장 완료: (119702, 32)


In [9]:
#오퍼 타입별 평균 퍼널 지표(확인율,전환율,확인→완료율,확인 없이 완료)
offer_metrics.groupby('offer_type')[[
    'view_rate',
    'conversion_rate',
    'viewed_to_completed_rate',
    'completed_wo_view_rate'
]].mean()

,view_rate,conversion_rate,viewed_to_completed_rate,completed_wo_view_rate
offer_type,,,,
bogo,0.823944,0.569017,0.418614,0.084787
discount,0.689806,0.633813,0.571196,0.126125
informational,0.645489,0.000000,0.000000,0.000000


## 7. 결론: 세그먼트 전략의 한계

소득·연령·성별·가입기간 기준으로 오퍼 반응 차이 분석 완료  
→ 세그먼트 간 차이는 존재하지만 **일관된 타겟팅 규칙 도출 실패**

문제:
"누가 반응하는가"를 묻고 있었지만  
→ 실제 질문은 **"누가 오퍼 없이 구매하는가"**였다

→ 다음 단계: 행동 패턴 기반 루틴형 고객 식별 (03)

---